# Energy Real-Data Results / Energy 真实数据结果

This notebook consolidates the formal energy real-data suite: CMDL, plain-LSTM baseline, and the three core ablation variants.
本 notebook 汇总 energy 真实数据正式实验：CMDL、plain-LSTM baseline，以及三个核心 ablation 变体。

## Sections / 结构

- Setup: repo paths, imports, and the single configuration cell.
- Optional direct-run: programmatic training hooks for CMDL, baseline, and ablation.
- Unified comparison: build normalized run-level tables for forecasting and lag/proxy diagnostics.
- Seed summaries and plots: aggregate the formal CMDL versus plain-LSTM runs and export figures.
- Prediction snapshots: compare canonical CMDL and baseline runs on representative entities.
- Verdict log: summarize what the energy domain currently supports.

## Defaults / 默认行为

- The active plan is `formal_target`.
- Direct-run switches are opt-in and default to `False`.
- The notebook reads from `data/energy/raw/energy_wgi_merged.csv`.
- The standardized target name `co2_per_unit_energy` is currently sourced from OWID `carbon_intensity_elec`.
- Outputs are organized under `outputs/notebook_energy/<plan>/...`.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

current = Path.cwd().resolve()
repo_root = next(
    (
        path
        for path in [current, *current.parents]
        if (path / "config").exists() and (path / "data").exists() and (path / "experiments").exists()
    ),
    None,
)

if repo_root is None:
    raise RuntimeError(f"Could not find repo root from {current}")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"repo_root = {repo_root}")

In [ ]:
import importlib

from evaluation import energy_comparison as energy_comparison_module

energy_comparison_module = importlib.reload(energy_comparison_module)
build_energy_comparison = energy_comparison_module.build_energy_comparison
build_task_table = energy_comparison_module.build_task_table
build_interpretability_table = energy_comparison_module.build_interpretability_table

PRESET_CONFIGS = {
    "quick_check": {"epochs": 3, "patience": 1, "log_every": 1},
    "notebook_medium": {"epochs": 30, "patience": 8, "log_every": 5},
    "formal_target": {"epochs": 120, "patience": 20, "log_every": 20},
}

ACTIVE_PLAN = "formal_target"
SEEDS = [0, 1, 2]
ABLATION_SEEDS = [0]
CANONICAL_SEED = SEEDS[0]
RUN_CMDL = False
RUN_BASELINE = False
RUN_ABLATIONS = False
DISABLE_MLFLOW = True
TREATMENT_COLUMN = "renewables_share_energy"
TARGET_COLUMN = "co2_per_unit_energy"
TARGET_SOURCE_NOTE = "OWID carbon_intensity_elec standardized as co2_per_unit_energy"

DATA_PATH = repo_root / "data" / "energy" / "raw" / "energy_wgi_merged.csv"
OUTPUT_ROOT = repo_root / "outputs" / "notebook_energy" / ACTIVE_PLAN
CMDL_OUTPUT_DIR = OUTPUT_ROOT / "cmdl"
BASELINE_OUTPUT_DIR = OUTPUT_ROOT / "plain_lstm"
ABLATION_OUTPUT_DIR = OUTPUT_ROOT / "ablation"
COMPARISON_DIR = OUTPUT_ROOT / "comparison"
PLOTS_DIR = OUTPUT_ROOT / "comparison_plots"

for path in [OUTPUT_ROOT, CMDL_OUTPUT_DIR, BASELINE_OUTPUT_DIR, ABLATION_OUTPUT_DIR, COMPARISON_DIR, PLOTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

COMMON_ARGS = {
    "csv_path": str(DATA_PATH),
    "year_start": 1996,
    "year_end": 2023,
    "train_end_year": 2011,
    "val_end_year": 2017,
    "treatment_column": TREATMENT_COLUMN,
    "target_column": TARGET_COLUMN,
    "feature_bundle": "minimal",
    "max_missing_share": 0.15,
    "lr": 1e-3,
    "lambda_r": 0.1,
    "temperature": 1.0,
    "lag_bias_strength": 1.0,
    "grad_clip": 1.0,
    "device": "auto",
    "disable_mlflow": DISABLE_MLFLOW,
}

plan_table = pd.DataFrame.from_dict(PRESET_CONFIGS, orient="index")
notebook_switches = pd.DataFrame(
    {
        "value": [
            ACTIVE_PLAN,
            DATA_PATH,
            TARGET_SOURCE_NOTE,
            CMDL_OUTPUT_DIR,
            BASELINE_OUTPUT_DIR,
            ABLATION_OUTPUT_DIR,
            COMPARISON_DIR,
            PLOTS_DIR,
            ", ".join(str(seed) for seed in SEEDS),
            ", ".join(str(seed) for seed in ABLATION_SEEDS),
            CANONICAL_SEED,
            RUN_CMDL,
            RUN_BASELINE,
            RUN_ABLATIONS,
            DISABLE_MLFLOW,
        ]
    },
    index=[
        "ACTIVE_PLAN",
        "DATA_PATH",
        "TARGET_SOURCE_NOTE",
        "CMDL_OUTPUT_DIR",
        "BASELINE_OUTPUT_DIR",
        "ABLATION_OUTPUT_DIR",
        "COMPARISON_DIR",
        "PLOTS_DIR",
        "SEEDS",
        "ABLATION_SEEDS",
        "CANONICAL_SEED",
        "RUN_CMDL",
        "RUN_BASELINE",
        "RUN_ABLATIONS",
        "DISABLE_MLFLOW",
    ],
)

display(plan_table)
display(pd.Series({**COMMON_ARGS, **PRESET_CONFIGS[ACTIVE_PLAN]}, name="value").to_frame())
display(notebook_switches)
print(f"ACTIVE_PLAN = {ACTIVE_PLAN}")
print(f"OUTPUT_ROOT = {OUTPUT_ROOT}")

In [ ]:
from argparse import Namespace

if RUN_CMDL or RUN_BASELINE or RUN_ABLATIONS:
    from experiments import run_energy as run_energy_module
    from experiments import run_energy_ablation as run_energy_ablation_module
    from experiments import run_energy_lstm_baseline as run_energy_lstm_baseline_module

    run_energy_module = importlib.reload(run_energy_module)
    run_energy_ablation_module = importlib.reload(run_energy_ablation_module)
    run_energy_lstm_baseline_module = importlib.reload(run_energy_lstm_baseline_module)

    run_cmdl_experiment = run_energy_module.run_experiment
    run_baseline_experiment = run_energy_lstm_baseline_module.run_experiment
    run_ablation_suite = run_energy_ablation_module.run_suite
else:
    run_cmdl_experiment = None
    run_baseline_experiment = None
    run_ablation_suite = None

cmdl_summaries: list[dict[str, object]] = []
baseline_summaries: list[dict[str, object]] = []
ablation_summary_frame = pd.DataFrame()
ablation_aggregated_frame = pd.DataFrame()

if RUN_CMDL:
    for seed in SEEDS:
        cmdl_args = Namespace(
            **COMMON_ARGS,
            **PRESET_CONFIGS[ACTIVE_PLAN],
            seed=seed,
            output_dir=str(CMDL_OUTPUT_DIR),
            experiment_name=f"E3_energy_cmdl_seed{seed}",
            smoke=ACTIVE_PLAN == "quick_check",
        )
        cmdl_summary = run_cmdl_experiment(cmdl_args)
        cmdl_summaries.append(cmdl_summary)
        print(
            f"CMDL seed={seed} test_r2={cmdl_summary['metrics']['test']['r2']:.4f} "
            f"test_mae={cmdl_summary['metrics']['test']['mae']:.4f}"
        )
else:
    print("RUN_CMDL = False; skipping direct CMDL training.")

if RUN_BASELINE:
    for seed in SEEDS:
        baseline_args = Namespace(
            **COMMON_ARGS,
            **PRESET_CONFIGS[ACTIVE_PLAN],
            seed=seed,
            output_dir=str(BASELINE_OUTPUT_DIR),
            experiment_name=f"E3_energy_lstm_seed{seed}",
            smoke=ACTIVE_PLAN == "quick_check",
        )
        baseline_summary = run_baseline_experiment(baseline_args)
        baseline_summaries.append(baseline_summary)
        print(
            f"Baseline seed={seed} test_r2={baseline_summary['metrics']['test']['r2']:.4f} "
            f"test_mae={baseline_summary['metrics']['test']['mae']:.4f}"
        )
else:
    print("RUN_BASELINE = False; skipping direct baseline training.")

if RUN_ABLATIONS:
    ablation_args = Namespace(
        **COMMON_ARGS,
        **PRESET_CONFIGS[ACTIVE_PLAN],
        variant="all",
        seeds=ABLATION_SEEDS,
        output_dir=str(ABLATION_OUTPUT_DIR),
        experiment_prefix="E3_energy_ablation",
        smoke=ACTIVE_PLAN == "quick_check",
    )
    ablation_summary_frame, ablation_aggregated_frame = run_ablation_suite(ablation_args)
    display(ablation_summary_frame)
    if not ablation_aggregated_frame.empty:
        display(ablation_aggregated_frame)
else:
    print("RUN_ABLATIONS = False; skipping direct ablation training.")

In [ ]:
comparison = build_energy_comparison(
    cmdl_root=CMDL_OUTPUT_DIR,
    baseline_root=BASELINE_OUTPUT_DIR,
    ablation_root=ABLATION_OUTPUT_DIR,
)
task_table = build_task_table(comparison)
interpretability_table = build_interpretability_table(comparison)

comparison.to_csv(COMPARISON_DIR / "energy_comparison.csv", index=False)
task_table.to_csv(COMPARISON_DIR / "energy_forecast_comparison.csv", index=False)
interpretability_table.to_csv(COMPARISON_DIR / "energy_interpretability_comparison.csv", index=False)

display(task_table)
display(interpretability_table)
print(comparison.groupby("family").size().to_string())

In [ ]:
seed_frame = comparison.loc[comparison["family"].isin(["cmdl", "plain_lstm"])].copy()
seed_summary = (
    seed_frame.groupby(["family", "display_name"], as_index=False)
    .agg(
        test_r2_mean=("test_r2", "mean"),
        test_r2_std=("test_r2", "std"),
        test_mae_mean=("test_mae", "mean"),
        test_mae_std=("test_mae", "std"),
        test_proxy_signal_r2_mean=("test_proxy_signal_r2", "mean"),
        test_proxy_signal_r2_std=("test_proxy_signal_r2", "std"),
        test_effective_kstar_proxy_spearman_rho_mean=("test_effective_kstar_proxy_spearman_rho", "mean"),
        test_effective_kstar_proxy_spearman_rho_std=("test_effective_kstar_proxy_spearman_rho", "std"),
        test_effective_kstar_std_mean=("test_effective_kstar_std", "mean"),
        test_effective_kstar_std_std=("test_effective_kstar_std", "std"),
    )
)
seed_summary.to_csv(COMPARISON_DIR / "energy_seed_summary.csv", index=False)
display(seed_summary)

canonical = {}
for family in ["cmdl", "plain_lstm"]:
    family_frame = comparison.loc[comparison["family"] == family].sort_values(
        ["test_r2", "test_effective_kstar_std"],
        ascending=[False, False],
        na_position="last",
    )
    if not family_frame.empty:
        row = family_frame.iloc[0]
        canonical[family] = {
            "experiment": row["experiment"],
            "run_dir": row["run_dir"],
            "test_r2": None if pd.isna(row["test_r2"]) else float(row["test_r2"]),
            "test_mae": None if pd.isna(row["test_mae"]) else float(row["test_mae"]),
        }

with (COMPARISON_DIR / "canonical_runs.json").open("w", encoding="utf-8") as handle:
    json.dump(canonical, handle, indent=2, ensure_ascii=True)

display(pd.DataFrame(canonical).T)

In [ ]:
if comparison.empty:
    print("No formal energy results found; skipping plots.")
else:
    plot_frame = comparison.copy()
    plot_frame["run_label"] = plot_frame["display_name"] + " | seed " + plot_frame["seed"].astype(str)

    metrics_to_plot = [
        ("test_r2", "Forecast R2"),
        ("test_effective_kstar_std", "Effective k* std"),
        ("test_proxy_signal_r2", "Proxy signal R2"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for axis, (metric_name, title) in zip(axes, metrics_to_plot):
        subset = plot_frame.loc[plot_frame[metric_name].notna()].copy()
        axis.bar(subset["run_label"], subset[metric_name], color="#2a6f97")
        axis.set_title(title)
        axis.set_xlabel("run")
        axis.set_ylabel(metric_name)
        axis.tick_params(axis="x", rotation=25)
        axis.grid(alpha=0.25)
    fig.tight_layout()
    metrics_plot_path = PLOTS_DIR / "energy_formal_metrics.png"
    fig.savefig(metrics_plot_path, dpi=160, bbox_inches="tight")
    display(fig)
    plt.close(fig)

    canonical_cmdl = comparison.loc[comparison["family"] == "cmdl"].sort_values(
        ["test_r2", "test_effective_kstar_std"],
        ascending=[False, False],
        na_position="last",
    ).iloc[0]
    canonical_baseline = comparison.loc[comparison["family"] == "plain_lstm"].sort_values(
        ["test_r2", "test_effective_kstar_std"],
        ascending=[False, False],
        na_position="last",
    ).iloc[0]

    prediction_specs = {
        "CMDL": Path(canonical_cmdl["run_dir"]) / "predictions.csv",
        "Plain LSTM": Path(canonical_baseline["run_dir"]) / "predictions.csv",
    }
    prediction_frames = {label: pd.read_csv(path) for label, path in prediction_specs.items() if path.exists()}

    if not prediction_frames:
        print("Prediction files are missing; skipping prediction snapshots.")
    else:
        reference_frame = prediction_frames["CMDL"]
        if "k_star" in reference_frame.columns:
            entity_rank = reference_frame.groupby("entity_code", as_index=False)["k_star"].first().sort_values("k_star")
            entity_codes = list(dict.fromkeys(entity_rank["entity_code"].tolist()[:1] + entity_rank["entity_code"].tolist()[-1:]))
        else:
            entity_codes = sorted(reference_frame["entity_code"].unique().tolist())[:2]

        fig, axes = plt.subplots(len(entity_codes), 1, figsize=(12, 4 * len(entity_codes)), sharex=True)
        if len(entity_codes) == 1:
            axes = [axes]

        for axis, entity_code in zip(axes, entity_codes):
            truth_frame = reference_frame[reference_frame["entity_code"] == entity_code]
            axis.plot(truth_frame["year"], truth_frame["y_true"], color="black", linewidth=2.0, label="y_true")
            for label, frame in prediction_frames.items():
                entity_frame = frame[frame["entity_code"] == entity_code]
                axis.plot(entity_frame["year"], entity_frame["y_pred"], linewidth=1.8, label=label)
            axis.set_title(f"Canonical energy runs | entity {entity_code}")
            axis.set_ylabel(TARGET_COLUMN)
            axis.grid(alpha=0.25)
            axis.legend()

        axes[-1].set_xlabel("year")
        fig.tight_layout()
        prediction_plot_path = PLOTS_DIR / "energy_prediction_snapshots.png"
        fig.savefig(prediction_plot_path, dpi=160, bbox_inches="tight")
        display(fig)
        plt.close(fig)

        display(
            comparison.loc[comparison["family"].isin(["cmdl", "plain_lstm"])][
                [
                    "display_name",
                    "seed",
                    "test_r2",
                    "test_mae",
                    "test_effective_kstar_proxy_spearman_rho",
                    "test_effective_kstar_std",
                    "test_proxy_signal_r2",
                ]
            ].sort_values(["display_name", "seed"])
        )

        print(f"Saved plots to {metrics_plot_path} and {prediction_plot_path}")

In [ ]:
result_rows: list[dict[str, object]] = []

if comparison.empty:
    print("No formal energy results found; cannot build verdict log.")
else:
    cmdl_rows = comparison.loc[comparison["family"] == "cmdl"].copy()
    plain_lstm_rows = comparison.loc[comparison["family"] == "plain_lstm"].copy()
    no_ac_rows = comparison.loc[comparison["variant"] == "no_ac_encoder"].copy()
    uniform_rows = comparison.loc[comparison["variant"] == "uniform_lag"].copy()
    no_recon_rows = comparison.loc[comparison["variant"] == "no_recon_regularization"].copy()

    cmdl_mean_r2 = float(cmdl_rows["test_r2"].mean())
    baseline_mean_r2 = float(plain_lstm_rows["test_r2"].mean())
    cmdl_mean_rho = float(cmdl_rows["test_effective_kstar_proxy_spearman_rho"].mean())
    baseline_mean_rho = float(plain_lstm_rows["test_effective_kstar_proxy_spearman_rho"].mean())
    cmdl_mean_kstar_std = float(cmdl_rows["test_effective_kstar_std"].mean())
    baseline_mean_kstar_std = float(plain_lstm_rows["test_effective_kstar_std"].mean())

    result_rows.extend(
        [
            {
                "stage": "contract",
                "question": "Which energy contract is being audited?",
                "result": "renewables_share_energy -> co2_per_unit_energy",
                "evidence": TARGET_SOURCE_NOTE,
            },
            {
                "stage": "forecast",
                "question": "Does CMDL beat the matched plain LSTM on mean test R2?",
                "result": "No clear forecast lift",
                "evidence": f"CMDL mean test_r2={cmdl_mean_r2:.4f}, Plain LSTM mean test_r2={baseline_mean_r2:.4f}.",
            },
            {
                "stage": "mechanism",
                "question": "Does CMDL learn a more dispersed effective lag pattern than the baseline?",
                "result": "Yes",
                "evidence": f"CMDL mean effective k* std={cmdl_mean_kstar_std:.4f} versus Plain LSTM {baseline_mean_kstar_std:.4f}.",
            },
            {
                "stage": "mechanism",
                "question": "Is lag-proxy alignment directionally cleaner under CMDL than the baseline?",
                "result": "Partially",
                "evidence": f"CMDL mean rho={cmdl_mean_rho:.4f} versus Plain LSTM {baseline_mean_rho:.4f}; signs are not yet uniformly stable.",
            },
            {
                "stage": "ablation",
                "question": "What happens when the AC encoder is removed?",
                "result": "Lag heterogeneity collapses",
                "evidence": f"No AC Encoder test effective k* std={float(no_ac_rows['test_effective_kstar_std'].iloc[0]):.6f}, proxy refit status={no_ac_rows['proxy_refit_status'].iloc[0]}.",
            },
            {
                "stage": "ablation",
                "question": "What happens under uniform lag?",
                "result": "Interpretability is fixed by construction",
                "evidence": f"Uniform Lag entropy={float(uniform_rows['test_effective_lag_entropy_mean'].iloc[0]):.4f} and effective k* std={float(uniform_rows['test_effective_kstar_std'].iloc[0]):.6f}.",
            },
            {
                "stage": "ablation",
                "question": "Does removing reconstruction regularization obviously help?",
                "result": "Not from the first formal run",
                "evidence": f"No Recon test_r2={float(no_recon_rows['test_r2'].iloc[0]):.4f}, proxy_signal_r2={float(no_recon_rows['test_proxy_signal_r2'].iloc[0]):.4f}.",
            },
            {
                "stage": "verdict",
                "question": "Current energy verdict",
                "result": "Energy currently supports the lag-heterogeneity machinery more clearly than forecast superiority.",
                "evidence": "This is a viable second real-data domain, but not yet a strong standalone forecast win.",
            },
        ]
    )

energy_result_log = pd.DataFrame(result_rows)
energy_result_log.to_csv(COMPARISON_DIR / "energy_result_log.csv", index=False)
display(energy_result_log)